In [1]:
"""51 Peg b: synthetic demonstrations of the orbital analysis (Mayor & Queloz 1995).
All data are SIMULATED from Table 1 parameters; the paper's raw RVs are not published in the text.
"""
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.patches import FancyArrowPatch
from astropy.timeseries import LombScargle

os.makedirs("out", exist_ok=True)
P = 4.2293          # d, Table 1
K = 59.0            # m/s, Table 1 (0.059 km/s)
SIG = 13.0          # m/s, ELODIE precision / r.m.s. residual
T0 = 797.773        # JD-2449000, epoch of max velocity, Table 1
rng = np.random.default_rng(51)
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
COLS = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
EPNAMES = ["Sep-Nov 94", "Jan-Feb 95", "Jul 95", "Sep 95"]


def model(t):
    return K * np.cos(2 * np.pi * (t - T0) / P)


# ---------- sampling pattern: 35 points over four epochs (approximate, JD-2449000) ----------
def make_times():
    e1 = np.sort(rng.uniform(598, 660, 5))
    e2 = np.sort(rng.uniform(730, 770, 6))
    n3 = 908 + np.arange(8)
    e3 = np.sort(np.concatenate([n3, 908 + rng.choice(8, 4, replace=False) + 0.15]))
    n4 = 960 + np.arange(8)
    e4 = np.sort(np.concatenate([n4, 960 + rng.choice(8, 4, replace=False) + 0.15]))
    t = np.concatenate([e1, e2, e3, e4])
    ep = np.array([0] * 5 + [1] * 6 + [2] * 12 + [3] * 12)
    return t, ep


T, EP = make_times()
V = model(T) + rng.normal(0, SIG, len(T))


# =========================== Animation 1: orbit -> RV curve ===========================
def anim_orbit():
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 5), dpi=80, gridspec_kw={"width_ratios": [1, 1.3]})
    R_p, R_s = 1.0, 0.28
    a1.set_xlim(-1.6, 1.6); a1.set_ylim(-1.95, 1.65); a1.set_aspect("equal"); a1.axis("off")
    a1.set_title("Orbit seen from above (wobble exaggerated ~600x)")
    th = np.linspace(0, 2 * np.pi, 300)
    a1.plot(R_p * np.cos(th), R_p * np.sin(th), color="0.8", lw=1)
    a1.plot(R_s * np.cos(th), R_s * np.sin(th), color="0.8", lw=1)
    a1.plot(0, 0, "k+", ms=9)
    star, = a1.plot([], [], "o", color="gold", ms=26, mec="orange", zorder=3)
    planet, = a1.plot([], [], "o", color="tab:blue", ms=11, zorder=3)
    a1.add_patch(FancyArrowPatch((0, -1.45), (0, -1.85), arrowstyle="<|-", mutation_scale=18, color="k"))
    a1.text(0, -1.95, "observer (Earth) is down here", ha="center", va="top", fontsize=10)
    a1.text(-1.55, 1.2, "star", color="darkorange", fontsize=10)
    a1.text(-1.55, 1.0, "planet", color="tab:blue", fontsize=10)
    readout = a1.text(0, 1.5, "", ha="center", fontsize=12, weight="bold")

    phis = np.linspace(0, 2, 201)
    vr = K * np.cos(2 * np.pi * phis)
    a2.set_xlim(0, 2); a2.set_ylim(-85, 85)
    a2.axhline(0, color="0.6", ls=":")
    a2.set_xlabel("Orbital phase (cycles)"); a2.set_ylabel("Star radial velocity (m/s)")
    a2.set_title("What the spectrograph measures: K = 59 m/s, P = 4.2293 d")
    line, = a2.plot([], [], color="tab:red", lw=2)
    dot, = a2.plot([], [], "o", color="k", ms=8)
    arrows = []

    def update(i):
        phi = phis[i]; t = 2 * np.pi * phi
        sx, sy = R_s * np.cos(t), R_s * np.sin(t)
        px, py = -R_p * np.cos(t), -R_p * np.sin(t)
        star.set_data([sx], [sy]); planet.set_data([px], [py])
        for a in arrows:
            a.remove()
        arrows.clear()
        v = K * np.cos(t)  # +y is away from the observer => positive = receding
        c = "tab:red" if v > 0 else "tab:blue"
        ar = FancyArrowPatch((sx, sy), (sx - 0.6 * np.sin(t), sy + 0.6 * np.cos(t)),
                             arrowstyle="-|>", mutation_scale=20, color=c, lw=2.5, zorder=4)
        a1.add_patch(ar); arrows.append(ar)
        if abs(v) < 3:
            word = "moving sideways, no Doppler shift"
        else:
            word = "receding, redshift" if v > 0 else "approaching, blueshift"
        readout.set_text(f"V_r = {v:+.0f} m/s  ({word})")
        readout.set_color(c)
        line.set_data(phis[: i + 1], vr[: i + 1]); dot.set_data([phi], [v])
        return []

    fig.tight_layout()
    update(25); fig.savefig("out/preview_orbit.png")
    ani = FuncAnimation(fig, update, frames=len(phis), interval=50)
    ani.save("out/anim1_orbit_to_rv_curve.gif", writer=PillowWriter(fps=20))
    plt.close(fig)


# =========================== Animation 2: data arrive -> periodogram sharpens ===========================
def anim_data():
    fig = plt.figure(figsize=(15, 4.8), dpi=70)
    ax1, ax2, ax3 = fig.subplots(1, 3)
    freq = np.linspace(1 / 40, 1 / 1.05, 15000)
    per = 1 / freq
    frames = list(range(5, 36)) + [35] * 10
    summary = {}

    def draw(n):
        for a in (ax1, ax2, ax3):
            a.cla()
            a.spines["top"].set_visible(False); a.spines["right"].set_visible(False)
        # time series
        for k in range(4):
            m = EP[:n] == k
            ax1.errorbar(T[:n][m], V[:n][m], SIG, fmt="o", color=COLS[k], ms=5, capsize=2, label=EPNAMES[k])
        ax1.axhline(0, color="0.7", ls=":")
        ax1.set_xlim(585, 985); ax1.set_ylim(-120, 120)
        ax1.set_xlabel("JD - 2449000 (days)"); ax1.set_ylabel("V_r (m/s)")
        ax1.set_title(f"Observations arrive epoch by epoch (N = {n})")
        ax1.legend(fontsize=8, loc="lower right", frameon=False)
        # periodogram
        ls = LombScargle(T[:n], V[:n], SIG)
        pw = ls.power(freq)
        b = freq[np.argmax(pw)]
        ax2.plot(per, pw, color="k", lw=1.2)
        ax2.axvline(P, color="tab:red", ls="--", lw=1)
        ax2.text(P * 1.05, 0.93, "true 4.23 d", color="tab:red", fontsize=9)
        ax2.set_xscale("log"); ax2.set_xlim(1, 40); ax2.set_ylim(0, 1.02)
        ax2.set_xticks([1, 2, 4, 8, 16, 32]); ax2.set_xticklabels(["1", "2", "4", "8", "16", "32"])
        ax2.set_xlabel("Trial period (days)"); ax2.set_ylabel("Lomb-Scargle power")
        ax2.set_title("Periodogram of data so far")
        # phase fold at current best period
        ph = ((T[:n] - T0) * b) % 1
        for k in range(4):
            m = EP[:n] == k
            ax3.errorbar(ph[m], V[:n][m], SIG, fmt="o", color=COLS[k], ms=5, capsize=2)
        tg = T0 + np.linspace(0, 1, 300) / b
        ax3.plot(np.linspace(0, 1, 300), ls.model(tg, b), color="k", lw=1.5)
        ax3.set_xlim(0, 1); ax3.set_ylim(-120, 120)
        ax3.set_xlabel("Phase"); ax3.set_ylabel("V_r (m/s)")
        ax3.set_title(f"Folded at best period = {1 / b:.3f} d")
        fig.tight_layout()
        if n == 35:
            mp = ls.model_parameters(b)
            summary["P"] = 1 / b
            summary["K"] = float(np.hypot(mp[1], mp[2]))

    draw(20); fig.savefig("out/preview_data.png")
    ani = FuncAnimation(fig, draw, frames=frames, interval=300)
    ani.save("out/anim2_data_and_periodogram.gif", writer=PillowWriter(fps=3))
    plt.close(fig)
    print("Anim2 final LS best period (d):", summary.get("P"), " amplitude (m/s):", summary.get("K"))


# =========================== Figure 3: gamma drift vs extra free parameters ===========================
def design(t, ep, per_epoch):
    ph = 2 * np.pi * (t - T0) / P
    cols = [np.cos(ph), np.sin(ph)]
    if per_epoch:
        cols += [(ep == k).astype(float) for k in range(4)]
    else:
        cols += [np.ones_like(t)]
    return np.column_stack(cols)


def fit(t, v, ep, per_epoch):
    A = design(t, ep, per_epoch)
    p, *_ = np.linalg.lstsq(A, v, rcond=None)
    res = v - A @ p
    return p, res, np.sqrt(np.mean(res ** 2))


def fig_gamma():
    drift = np.array([-25.0, 15.0, -10.0, 20.0])  # m/s offsets per epoch (assumed, illustrative)
    v_drift = model(T) + drift[EP] + rng.normal(0, SIG, len(T))
    p_g, r_g, rms_g = fit(T, v_drift, EP, False)
    p_e, r_e, rms_e = fit(T, v_drift, EP, True)
    ph_all = ((T - T0) / P) % 1
    # Monte Carlo control: NO drift at all
    rg, re = [], []
    for _ in range(3000):
        vv = model(T) + rng.normal(0, SIG, len(T))
        rg.append(fit(T, vv, EP, False)[2]); re.append(fit(T, vv, EP, True)[2])
    rg, re = np.array(rg), np.array(re)

    fig, ax = plt.subplots(1, 3, figsize=(15, 4.6), dpi=80)
    for a in ax[:2]:
        a.set_xlim(0, 1); a.set_ylim(-120, 120); a.set_xlabel("Phase"); a.set_ylabel("V_r (m/s)")
    tg = np.linspace(0, 1, 300)
    for k in range(4):
        m = EP == k
        ax[0].errorbar(ph_all[m], v_drift[m], SIG, fmt="o", color=COLS[k], ms=5, capsize=2, label=EPNAMES[k])
        ax[1].errorbar(ph_all[m], v_drift[m] - p_e[2 + k] + 0 * 1, SIG, fmt="o", color=COLS[k], ms=5, capsize=2)
    ax[0].plot(tg, p_g[0] * np.cos(2 * np.pi * tg) + p_g[1] * np.sin(2 * np.pi * tg) + p_g[2], "k")
    ax[1].plot(tg, p_e[0] * np.cos(2 * np.pi * tg) + p_e[1] * np.sin(2 * np.pi * tg), "k")
    ax[0].set_title(f"Epoch offsets present, ONE gamma\nr.m.s. residual = {rms_g:.0f} m/s")
    ax[1].set_title(f"Same data, gamma free PER EPOCH\nr.m.s. residual = {rms_e:.0f} m/s")
    ax[0].legend(fontsize=8, frameon=False, loc="lower right")
    ax[2].hist(rg, bins=40, alpha=0.6, label="one gamma", color="tab:gray")
    ax[2].hist(re, bins=40, alpha=0.6, label="gamma per epoch", color="tab:purple")
    ax[2].axvline(24, color="tab:red", ls="--"); ax[2].text(24.3, ax[2].get_ylim()[1] * 0.85, "paper's\nfirst fit: 24", color="tab:red", fontsize=9)
    ax[2].set_xlabel("r.m.s. residual (m/s)"); ax[2].set_ylabel("Count (3000 trials)")
    ax[2].set_title("Control: NO drift, pure 13 m/s noise\nextra parameters alone barely help")
    ax[2].legend(fontsize=9, frameon=False, loc="upper left")
    fig.tight_layout()
    fig.savefig("out/fig3_gamma_drift_vs_free_parameters.png")
    plt.close(fig)
    print(f"Drift case: rms one-gamma={rms_g:.1f}, per-epoch={rms_e:.1f}")
    print(f"No-drift control: one-gamma mean rms={rg.mean():.1f}, per-epoch mean rms={re.mean():.1f}, "
          f"max per-epoch rms in 3000 trials={re.max():.1f}, frac(one-gamma rms>=24)={(rg >= 24).mean():.4f}")


if __name__ == "__main__":
    print("N points:", len(T))
    anim_orbit()
    anim_data()
    fig_gamma()
    print("done")

N points: 35
Anim2 final LS best period (d): 4.230205682283658  amplitude (m/s): 60.849022667433374
Drift case: rms one-gamma=19.4, per-epoch=14.3
No-drift control: one-gamma mean rms=12.3, per-epoch mean rms=11.7, max per-epoch rms in 3000 trials=18.2, frac(one-gamma rms>=24)=0.0000
done
